In [129]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [130]:
# TODO: Add post-processing to actions, allowing for discretization of continuous actions
# TODO: Make tensorboard logging optional and move all functions to a separate file
# TODO: Add video logging to tensorboard
# TODO: Refactor initialization
# TODO: Verify unimix values
# TODO: Properly implement ratio
# TODO: Implement model saving and loading, plus buffer maybe?
# TODO: Implement https://rlgym.org/
# TODO: Add more tests
# TODO: Allow vectorized buffer storage and sampling

In [131]:
import copy
import time

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.utils.tensorboard

import fishyrl as frl

In [132]:
# Create environment
num_envs = 8
env_name = 'BipedalWalker-v3'
envs = gym.vector.AsyncVectorEnv([lambda: gym.make(env_name) for _ in range(num_envs)])

# Tested
# CartPole-v1: Begins converging to optimal after 10 minutes w/ 512 EOD, 2 NB, 512 DD, 32 SD, 512 DD, and sequence length 10
# MountainCar-v0: Doesn't get an initial reward to work from with same params as CartPole-v1
# BipedalWalker-v3: Learns to stand

In [133]:
# Initialize model
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

OBS_DIM = envs.observation_space.shape[1]
# MODEL_ACTIONS = [frl.actions.DiscreteAction(2)]  # CartPole-v1
# MODEL_ACTIONS = [frl.actions.DiscreteAction(3)]  # MountainCar-v0
MODEL_ACTIONS = [frl.actions.ContinuousActions(4, clip=1)]  # BipedalWalker-v3
MODEL_ACTION_OUTPUT_DIMS = [action.output_dim for action in MODEL_ACTIONS]
ACTION_DIM = sum(MODEL_ACTION_OUTPUT_DIMS)

EMBEDDED_OBS_DIM = 512  # 1024
NUM_BLOCKS = 2  # 5
DENSE_DIM = 512  # 1024

CATEGORICAL_BINS = 32
REWARD_BINS = 255

STOCHASTIC_DIM = 32  # NOTE: This is multiplied by `CATEGORICAL_BINS`
DETERMINISTIC_DIM = 512  # 4096

# Seed
torch.random.manual_seed(42)

# Encoder-decoder models
encoder_model = frl.models.MLPEncoder(OBS_DIM, EMBEDDED_OBS_DIM, num_blocks=NUM_BLOCKS, hidden_dim=DENSE_DIM).to(DEVICE)
decoder_model = frl.models.MLPDecoder(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, OBS_DIM, num_blocks=NUM_BLOCKS, hidden_dim=DENSE_DIM).to(DEVICE)

# RSSM models
recurrent_model = frl.models.RecurrentModel(STOCHASTIC_DIM * CATEGORICAL_BINS + ACTION_DIM, DETERMINISTIC_DIM).to(DEVICE)
representation_model = frl.models.MLP(EMBEDDED_OBS_DIM + DETERMINISTIC_DIM, STOCHASTIC_DIM * CATEGORICAL_BINS).to(DEVICE)
transition_model = frl.models.MLP(DETERMINISTIC_DIM, STOCHASTIC_DIM * CATEGORICAL_BINS).to(DEVICE)
rssm_model = frl.models.RSSM(recurrent_model, representation_model, transition_model, CATEGORICAL_BINS).to(DEVICE)

# Reward and continue models
reward_model = frl.models.MLP(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, REWARD_BINS, NUM_BLOCKS * [DENSE_DIM]).to(DEVICE)
continue_model = frl.models.MLP(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, 1, NUM_BLOCKS * [DENSE_DIM]).to(DEVICE)

# Actor and critic models
actor_model = frl.models.Actor(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, MODEL_ACTIONS).to(DEVICE).apply(frl.utilities.init_weights)
critic_model = frl.models.MLP(STOCHASTIC_DIM * CATEGORICAL_BINS + DETERMINISTIC_DIM, REWARD_BINS, NUM_BLOCKS * [DENSE_DIM]).to(DEVICE).apply(frl.utilities.init_weights)
target_critic_model = copy.deepcopy(critic_model)

# Hafner weight initialization
actor_model._model._model[-1].apply(frl.utilities.uniform_init_weights(1.))
critic_model._model[-1].apply(frl.utilities.uniform_init_weights(0.))
rssm_model._transition_model._model[-1].apply(frl.utilities.uniform_init_weights(1.))
rssm_model._representation_model._model[-1].apply(frl.utilities.uniform_init_weights(1.))
reward_model._model[-1].apply(frl.utilities.uniform_init_weights(0.))
continue_model._model[-1].apply(frl.utilities.uniform_init_weights(1.))
decoder_model._model._model[-1].apply(frl.utilities.uniform_init_weights(1.))

# Buffer
buffer = frl.buffers.IndependentVectorizedBuffer(num_envs, 10**6)  # Buffer size of 1M

# Lambda value normalizer
lambda_normalizer = frl.utilities.MovingMinMaxScaler()

# Optimizers
# NOTE: All use Adam optimizer, https://github.com/Eclectic-Sheep/sheeprl/blob/33b636681fd8b5340b284f2528db8821ab8dcd0b/sheeprl/configs/algo/dreamer_v3.yaml
world_params = (
    list(encoder_model.parameters()) + list(decoder_model.parameters())
    + list(rssm_model.parameters()) + list(reward_model.parameters())
    + list(continue_model.parameters()))
world_optimizer = torch.optim.Adam(world_params, lr=1e-4, eps=1e-8)
actor_optimizer = torch.optim.Adam(actor_model.parameters(), lr=8e-5, eps=1e-5)
critic_optimizer = torch.optim.Adam(critic_model.parameters(), lr=8e-5, eps=1e-5)

# Tensorboard writer
folder_name = f'{env_name}_{time.strftime("%Y-%m-%d_%H-%M-%S")}'
writer = torch.utils.tensorboard.SummaryWriter(f'./runs/{folder_name}')

In [ ]:
environment_steps = 10**6  # Guessing
start_train_step = num_envs * 1024  # TODO: In the case of loading a checkpoint, don't randomly sample, but instead use the model until the starting training step
gradient_steps = num_envs  # Normally calculated with Ratio, but this is a ratio of 1
batch_size = 16  # Might need to refine, based on paper
sequence_length = 64  # 10  # Might need to refine, based on paper
imagination_horizon = 15
tau = .02  # Target critic update rate

def train(batch: dict[str, torch.Tensor], environment_step: int) -> None:
    """Single-step training function."""
    # Infer batch size and sequence length (batch, sequence_length, ...)
    batch_size, sequence_length = batch['obs'].shape[:2]

    # Embed observations
    embedded_obs = encoder_model(batch['obs'])

    # Initialize storage
    hidden_states = []
    priors = []
    priors_logits = []
    posteriors = []
    posteriors_logits = []

    # SheepRL does this, but creates an issue with autograd and doesn't appear to be faster
    # hidden_states = torch.empty((batch_size, sequence_length, DETERMINISTIC_DIM))
    # priors = torch.empty((batch_size, sequence_length, STOCHASTIC_DIM * CATEGORICAL_BINS))
    # priors_logits = torch.empty((batch_size, sequence_length, STOCHASTIC_DIM * CATEGORICAL_BINS))
    # posteriors = torch.empty((batch_size, sequence_length, STOCHASTIC_DIM * CATEGORICAL_BINS))
    # posteriors_logits = torch.empty((batch_size, sequence_length, STOCHASTIC_DIM * CATEGORICAL_BINS))

    # Compute model outputs for each time step, starting with initial recurrent state and posteriors
    for i in range(sequence_length):
        # Run through recurrent model
        ret = rssm_model(
            batch['actions'][:, i-1] if i > 0 else None,  # Use the action from the previous step, like in the environment loop
            posteriors[i-1] if i > 0 else None,
            hidden_states[i-1] if i > 0 else None,
            embedded_obs[:, i],
            batch['terminations'][:, i-1] | batch['truncations'][:, i-1] if i > 0 else None,  # Get initializations using result of previous step
            batch_dim=batch_size,
        )
        hidden_states.append(ret['hidden_state'])
        priors.append(ret['prior'])
        priors_logits.append(ret['prior_logits'])
        posteriors.append(ret['posterior'])
        posteriors_logits.append(ret['posterior_logits'])

        # SheepRL assignment approach
        # TODO: Figure out why this doesn't throw an error in SheepRL (It does right now)
        # hidden_states[:, i] = ret['hidden_state']
        # priors[:, i] = ret['prior']
        # priors_logits[:, i] = ret['prior_logits']
        # posteriors[:, i] = ret['posterior']
        # posteriors_logits[:, i] = ret['posterior_logits']

    # Concatenate returned tensors
    hidden_states = torch.stack(hidden_states, dim=1)
    priors = torch.stack(priors, dim=1)
    priors_logits = torch.stack(priors_logits, dim=1)
    posteriors = torch.stack(posteriors, dim=1)
    posteriors_logits = torch.stack(posteriors_logits, dim=1)

    # Compute predicted observations, rewards, and continues
    pred_obs = decoder_model(torch.cat((posteriors, hidden_states), dim=-1))
    pred_rewards = reward_model(torch.cat((posteriors, hidden_states), dim=-1))
    pred_continues = continue_model(torch.cat((posteriors, hidden_states), dim=-1))

    # Compute MSE loss for reconstructed observations (po)
    # observation_loss = frl.losses.mse_loss(pred_obs, XXX, dims=2)  # CNN
    observation_loss = frl.losses.mse_loss(pred_obs, frl.distributions.symlog(batch['obs'])).clip(min=1e-8)  # MLP

    # Compute reward loss (pr)
    dist_pred_rewards = frl.distributions.TwoHot(pred_rewards)  # NOTE: Default reward range is -20, 20
    reward_loss = -dist_pred_rewards.log_prob(batch['rewards'])  # TODO: Verify that this is the correct timestep, same with continues

    # Compute continue loss (pc)
    dist_pred_continues = torch.distributions.Bernoulli(logits=pred_continues.squeeze(-1))
    continues = 1 - 1. * batch['terminations']
    continue_loss = -dist_pred_continues.log_prob(continues)

    # Reshape priors and posteriors
    posteriors_logits = posteriors_logits.reshape([*posteriors_logits.shape[:-1], -1, CATEGORICAL_BINS])
    priors_logits = priors_logits.reshape([*priors_logits.shape[:-1], -1, CATEGORICAL_BINS])

    # KL Balancing
    dynamic_loss = torch.distributions.kl_divergence(
        torch.distributions.Independent(torch.distributions.OneHotCategoricalStraightThrough(logits=posteriors_logits.detach()), 1),
        torch.distributions.Independent(torch.distributions.OneHotCategoricalStraightThrough(logits=priors_logits), 1))
    representation_loss = torch.distributions.kl_divergence(
        torch.distributions.Independent(torch.distributions.OneHotCategoricalStraightThrough(logits=posteriors_logits), 1),
        torch.distributions.Independent(torch.distributions.OneHotCategoricalStraightThrough(logits=priors_logits.detach()), 1))
    free_nats = torch.tensor(1.)
    dynamic_loss = torch.maximum(dynamic_loss, free_nats)  # Free nats
    representation_loss = torch.maximum(representation_loss, free_nats)  # Free nats
    kl_loss = .5 * dynamic_loss + .1 * representation_loss

    # Total loss
    reconstruction_loss = (1 * kl_loss + observation_loss + reward_loss + 10 * continue_loss).mean()

    # Log to tensorboard
    writer.add_scalar('Loss/World', reconstruction_loss.item(), environment_step)
    writer.add_scalar('World/Observation', observation_loss.mean().item(), environment_step)
    writer.add_scalar('World/Reward', reward_loss.mean().item(), environment_step)
    writer.add_scalar('World/Continue', continue_loss.mean().item(), environment_step)
    writer.add_scalar('World/Dynamic', dynamic_loss.mean().item(), environment_step)
    writer.add_scalar('World/Representation', representation_loss.mean().item(), environment_step)

    # Step world model
    world_optimizer.zero_grad()
    reconstruction_loss.backward()
    nn.utils.clip_grad_norm_(world_params, max_norm=1000.)
    world_optimizer.step()

    # Behavior learning
    # imagined_prior = posteriors.detach().reshape(-1, posteriors.shape[-1])
    # imagined_hidden_state = hidden_states.detach().reshape(-1, hidden_states.shape[-1])
    imagined_prior, imagined_hidden_state = posteriors.detach(), hidden_states.detach()

    # Initialize storage
    imagined_trajectories = [torch.cat((imagined_prior, imagined_hidden_state), dim=-1)]
    imagined_actions = []

    # Imagine for `imagination_horizon` steps
    for _ in range(imagination_horizon):
        # Compute action based on previous prior + hidden state
        actions, action_distributions = actor_model(imagined_trajectories[-1])

        # Imagine future prior
        ret = rssm_model(actions, imagined_prior, imagined_hidden_state)
        imagined_prior = ret['prior']
        imagined_hidden_state = ret['hidden_state']

        # Record
        imagined_actions.append(actions)
        imagined_trajectories.append(torch.cat((imagined_prior, imagined_hidden_state), dim=-1))

    # Stack to (batch, sequence, horizon(-1), features)
    imagined_trajectories = torch.stack(imagined_trajectories, dim=2)
    imagined_actions = torch.stack(imagined_actions, dim=2)

    # Compute predicted rewards, values, and continues
    pred_rewards = reward_model(imagined_trajectories)
    pred_rewards = frl.distributions.TwoHot(pred_rewards).mean
    pred_values = critic_model(imagined_trajectories)
    pred_values = frl.distributions.TwoHot(pred_values).mean
    pred_continues = continue_model(imagined_trajectories[:, :, 1:])
    pred_continues = torch.distributions.Bernoulli(logits=pred_continues.squeeze(-1)).mode
    pred_continues = torch.cat((continues.unsqueeze(-1), pred_continues), dim=-1)  # Use actual continues where possible

    # Compute lambda values
    # NOTE: These are not advantages
    # https://github.com/Eclectic-Sheep/sheeprl/blob/33b636681fd8b5340b284f2528db8821ab8dcd0b/sheeprl/algos/dreamer_v3/utils.py#L66
    # https://github.com/danijar/dreamerv3/blob/b65cf81a6fb13625af8722127459283f899a35d9/dreamerv3/agent.py#L482
    lambda_values = [pred_values[:, :, -1]]  # Add bootstrapping value
    gamma = lmbda = .95
    interm = pred_rewards[:, :, 1:] + pred_continues[:, :, 1:] * gamma * pred_values[:, :, 1:] * (1 - lmbda)
    for i in range(pred_rewards[:, :, 1:].shape[2] - 1, -1, -1):
        lambda_values.append( interm[:, :, i] + pred_continues[:, :, 1:][:, :, i] * gamma * lmbda * lambda_values[-1] )
    lambda_values = torch.stack(lambda_values[:0:-1], dim=2)

    # Using previous values as baseline, compute advantage
    baseline = pred_values[:, :, :-1]
    norm_low, norm_range = lambda_normalizer(lambda_values)
    normalized_lambda_values = (lambda_values - norm_low) / norm_range
    normalized_baseline = (baseline - norm_low) / norm_range
    advantage = normalized_lambda_values - normalized_baseline

    # Compute discounts based on horizon, and disregard after non-continues
    horizon_discount = torch.cumprod(pred_continues[:, :, :-1] * gamma, dim=2) / gamma
    horizon_discount = horizon_discount.detach()

    # Compute objective by summing action log probabilities
    # TODO: Maybe this could be computed earlier? We could do it in the imagination loop, but it would propagate gradients throughout
    #       the horizon on the side of the distribution, which is not in the implementation.
    _, action_distributions = actor_model(imagined_trajectories[:, :, :-1].detach())
    objective = 0
    for dist, action in zip(action_distributions, imagined_actions.split(MODEL_ACTION_OUTPUT_DIMS, dim=-1)):
        objective += dist.log_prob(action.detach())
    objective = advantage.detach() * objective

    # Compute entropy
    entropy = 3e-4 * sum([dist.entropy() for dist in action_distributions])

    # Compute actor loss
    actor_loss = - horizon_discount.detach() * (objective + entropy)
    actor_loss = actor_loss.mean()

    # Log to tensorboard
    writer.add_scalar('Loss/Actor', actor_loss.item(), environment_step)
    writer.add_scalar('Actor/Objective', - objective.mean().item(), environment_step)
    writer.add_scalar('Actor/Entropy', - entropy.mean().item(), environment_step)

    # Step actor model
    actor_optimizer.zero_grad()
    actor_loss.backward()
    nn.utils.clip_grad_norm_(actor_model.parameters(), max_norm=100.)
    actor_optimizer.step()

    # Compute predicted critic and target critic values
    # TODO: Could do this earlier, but the stored gradients may complicate backpropagation, time-wise.
    pred_values = critic_model(imagined_trajectories[:, :, :-1].detach())
    dist_pred_values = frl.distributions.TwoHot(pred_values)
    pred_target_values = frl.distributions.TwoHot(target_critic_model(imagined_trajectories[:, :, :-1].detach())).mean

    # Compute critic loss
    target_loss = - dist_pred_values.log_prob(lambda_values.detach())
    divergence_loss = - dist_pred_values.log_prob(pred_target_values.detach())  # Don't stray too far from target critic values
    critic_loss = target_loss + divergence_loss
    critic_loss = horizon_discount.detach() * critic_loss  # Discount based on horizon
    critic_loss = critic_loss.mean()

    # Log to tensorboard
    writer.add_scalar('Loss/Critic', critic_loss.item(), environment_step)
    writer.add_scalar('Critic/Target', target_loss.mean().item(), environment_step)
    writer.add_scalar('Critic/Divergence', divergence_loss.mean().item(), environment_step)

    # Step critic model
    critic_optimizer.zero_grad()
    critic_loss.backward()
    nn.utils.clip_grad_norm_(critic_model.parameters(), max_norm=100.)
    critic_optimizer.step()

    # Reset gradients
    world_optimizer.zero_grad(set_to_none=True)
    actor_optimizer.zero_grad(set_to_none=True)
    critic_optimizer.zero_grad(set_to_none=True)


def main() -> None:
    """Main training loop."""
    # Initialize variables
    actions = posteriors = hidden_states = initializations = None
    rewards, terminations, truncations = np.zeros(num_envs), np.zeros(num_envs, dtype=bool), np.zeros(num_envs, dtype=bool)
    # pred_rewards = pred_continues = 0

    # Tracking
    cumulative_rewards = np.zeros(num_envs)

    # Loop for specified number of iterations
    obs, info = envs.reset(seed=42)
    for environment_step in range(0, environment_steps, num_envs):
        ## Compute an environment step
        if environment_step >= start_train_step:
            with torch.no_grad():
                # Embed observation
                embedded_obs = encoder_model(torch.from_numpy(obs).to(DEVICE))  # TODO: GPU

                # Compute hidden states
                ret = rssm_model(
                    actions,
                    posteriors,
                    hidden_states,
                    embedded_obs,
                    initializations,
                    batch_dim=envs.num_envs,
                )
                hidden_states = ret['hidden_state']
                # priors = ret['prior']
                # prior_logits = ret['prior_logits']
                posteriors = ret['posterior']
                # posterior_logits = ret['posterior_logits']

                # Decode observation
                # pred_obs = decoder_model(torch.cat((posteriors, hidden_states), dim=-1))

                # Predict reward and continue
                # TODO: Double-check, but compute reward for the previous action
                # pred_rewards = reward_model(torch.cat((posteriors, hidden_states), dim=-1))
                # pred_continues = continue_model(torch.cat((posteriors, hidden_states), dim=-1))

                # Compute actions
                actions, action_distributions = actor_model(torch.cat((posteriors, hidden_states), dim=-1))

                # Sample final actions
                env_actions = frl.actions.simplify_actions(actions, MODEL_ACTIONS).detach().cpu().numpy()
                if env_actions.shape[-1] == 1: env_actions = env_actions.squeeze(-1)  # Remove when using >1 action spaces
        # Compute action randomly if not training yet
        else:
            sampled_actions = envs.action_space.sample().reshape(num_envs, -1)
            actions = frl.actions.construct_actions(torch.tensor(sampled_actions, dtype=torch.get_default_dtype()), MODEL_ACTIONS)
            env_actions = sampled_actions.squeeze(-1) if sampled_actions.shape[-1] == 1 else sampled_actions  # Remove when using >1 action spaces

        # Record to buffer
        buffer.add({
            # Environment-related experiences
            'obs': obs,
            'rewards': rewards,
            'terminations': terminations,
            'truncations': truncations,
            # Predicted environment experiences
            # 'pred_rewards': pred_rewards.detach().cpu().numpy(),
            # 'pred_continues': pred_continues.detach().cpu().numpy(),
            # 'pred_obs': pred_obs.detach().cpu().numpy(),
            # RSSM experience
            # 'hidden_states': hidden_states.detach().cpu().numpy(),
            # 'priors': priors.detach().cpu().numpy(),
            # 'prior_logits': prior_logits.detach().cpu().numpy(),
            # 'posteriors': posteriors.detach().cpu().numpy(),
            # 'posterior_logits': posterior_logits.detach().cpu().numpy(),
            # Actions
            'actions': actions.detach().cpu().numpy(),
            # 'action_distributions': action_distributions,
        })

        # Step environment
        obs, rewards, terminations, truncations, infos = envs.step(env_actions)
        initializations = torch.tensor(terminations | truncations, dtype=torch.bool, device=DEVICE)

        # Progress indicator
        print('.', end='')
        if np.any(terminations | truncations): print()

        # Iterate and print rewards if done
        cumulative_rewards += rewards
        for i in range(num_envs):
            if terminations[i]:
                print(f'({environment_step}) Environment {i} terminated with cumulative reward: {cumulative_rewards[i]}')
                writer.add_scalar('Reward/Episode', cumulative_rewards[i], environment_step)
                cumulative_rewards[i] = 0
            elif truncations[i]:
                print(f'({environment_step}) Environment {i} truncated with cumulative reward: {cumulative_rewards[i]}')
                writer.add_scalar('Reward/Episode', cumulative_rewards[i], environment_step)
                cumulative_rewards[i] = 0

        ## Train
        if environment_step >= start_train_step:
            for _ in range(gradient_steps):
                # Sample batch of experiences from buffer and train
                batch = buffer.sample(batch_size, sequence_length=sequence_length)
                batch = frl.buffers.convert_samples_to_tensors(batch, device=DEVICE)

                # Iterate target critic model towards critic model
                update_tau = tau if gradient_steps > 0 else 1.
                for target_param, param in zip(target_critic_model.parameters(), critic_model.parameters()):
                    target_param.data.copy_(update_tau * param.data + (1 - update_tau) * target_param.data)

                # Run training step
                train(batch, environment_step)

In [ ]:
main()

...............................................
(368) Environment 4 terminated with cumulative reward: -108.99365703389049
..............
(480) Environment 2 terminated with cumulative reward: -102.8780922666192
.....................................
(776) Environment 7 terminated with cumulative reward: -116.63097455725074
.......................
(960) Environment 4 terminated with cumulative reward: -111.69439081847668
.......
(1016) Environment 2 terminated with cumulative reward: -111.92073924280703
...................................................................................
(1680) Environment 2 terminated with cumulative reward: -114.68180925399065
.............................................................
(2168) Environment 2 terminated with cumulative reward: -110.0387434931472
................................................................................
(2808) Environment 2 terminated with cumulative reward: -101.7555119106546
.......................................

In [ ]:
writer.close()

In [ ]:
{k: v.shape for k, v in buffer.sample(64, sequence_length=5).items()}